<a href="https://colab.research.google.com/github/Davron030901/Numpy/blob/main/MINI_TEST_WEEK_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import math
import time

#File I/O + Memory


In [ ]:
names = np.array(['Alice', 'Bob', 'Charlie'])
scores = np.array([[85, 90], [92, 88], [78, 95]])
dates = np.array(['2024-01-01', '2024-02-01'], dtype='datetime64')

print(f"Names: {names}")
print(f"Scores: {scores}")
print(f"Dates: {dates}")

Names: ['Alice' 'Bob' 'Charlie']
Scores: [[85 90]
 [92 88]
 [78 95]]
Dates: ['2024-01-01' '2024-02-01']


In [ ]:
np.savez_compressed('data.npz', names=names, scores=scores, dates=dates)

In [ ]:
load = np.load('data.npz')

In [ ]:
scores_load = load['scores']
print("Scores:", scores_load)

Scores: [[85 90]
 [92 88]
 [78 95]]


In [ ]:
memory = scores_load.nbytes
print(f"Memory: {memory} bites")

Memory: 48 bites


In [ ]:
view = scores_load[0]
print("View:", view)

View: [85 90]


In [ ]:
view = np.array(10)
print("View:", view)
print("Scores:", scores_load)

View: 10
Scores: [[85 90]
 [92 88]
 [78 95]]


In [ ]:
print("Equal:", np.array_equal(scores_load, scores))

Equal: True


#Memory Optimization

In [ ]:
arr = np.random.rand(1000, 1000)
print("Shape:", arr.shape)
print(f"Memory:{arr.nbytes/ (1024 ** 2):.2f}MB")

Shape: (1000, 1000)
Memory:7.63MB


BAD approach

In [ ]:
copy = arr.copy()
mult_20 = copy * 20
print("Mult shape:", mult_20.shape)
print(f"Memory:{mult_20.nbytes/ (1024 ** 2):.2f}MB")

Mult shape: (1000, 1000)
Memory:7.63MB


GOOD approach

In [ ]:
view = arr[::2]
print("View shape:", view.shape)

View shape: (500, 1000)


In [ ]:
mult_20 = view * 20
print("Mult shape:", mult_20.shape)
print(f"Memory:{mult_20.nbytes/ (1024 ** 2):.2f}MB")

Mult shape: (500, 1000)
Memory:3.81MB


Compare memory usage

In [ ]:
print(f"Memory Copy + Arr / View: {((arr.nbytes/ (1024 ** 2))+(copy.nbytes/ (1024 ** 2))) / (view.nbytes/ (1024 ** 2)):.2f}x useful ")

Memory Copy + Arr / View: 4.00x useful 


In [ ]:
difference_memory = (arr.nbytes + copy.nbytes) - (view.nbytes)
print(f"Memory Difference: {difference_memory/ (1024 ** 2):.2f}MB")

Memory Difference: 11.44MB


In [ ]:
print(f"View base: {view.base is arr}")
print(f"Copy base: {copy.base is arr}")

View base: True
Copy base: False


#Performance - Vectorization

In [ ]:
start = time.time()
result_loop = []
for i in range(10000):
  result_loop.append(math.sqrt(i**2 + i**2))
loop_time = time.time() - start
print(f"Loop time: {loop_time:.4f}s")

Loop time: 0.0015s


In [ ]:
start = time.time()
arr = np.arange(10000)
result_vec = np.sqrt(arr**2 + arr**2)
vec_time = time.time() - start
print(f"Vectorization time: {vec_time:.4f}s")

Vectorization time: 0.0007s


In [ ]:
print(f"Speed up:{loop_time / vec_time:.2f}x faster")

Speed up:2.25x faster


Verify results match (use np.allclose)

In [ ]:
match_result = np.allclose(result_loop, result_vec)
print(f"Match result: {match_result}")

Match result: True


#np.where() + Broadcasting

In [ ]:
matrix = np.array([[10, 20, 30], [40, 50, 60], [70, 80, 90]])
threshold = 50

print(f"Matrix:\n{matrix}")
print(f"Threshold: {threshold}")

Matrix:
[[10 20 30]
 [40 50 60]
 [70 80 90]]
Threshold: 50


In [ ]:
add_treshold = np.where(matrix > threshold, matrix * 2, matrix / 2)
print(f"Add threshold:\n{add_treshold}")

Add threshold:
[[  5.  10.  15.]
 [ 20.  25. 120.]
 [140. 160. 180.]]


Calculate using broadcasting: matrix - [5, 10, 15]

In [ ]:
calculate_broadcasting = add_treshold - [5, 10, 15]
print(f"Calculate using broadcasting:{calculate_broadcasting}")

Calculate using broadcasting:[[  0.   0.   0.]
 [ 15.  15. 105.]
 [135. 150. 165.]]


In [ ]:
large_100 = calculate_broadcasting[calculate_broadcasting > 100]
print(f"Large than 100: {large_100}")

Large than 100: [105. 135. 150. 165.]


In [ ]:
print(f"Count:{len(large_100)}")

Count:4


#Complete Integration

In [ ]:
response_times = np.array([120, 135, 145, 150, 155, 160, 165, 170, 500], dtype=np.int32)
timestamps = np.array([0, 1, 2, 3, 4, 5, 6, 7, 8], dtype=np.int32)

print(f"Response times:\n{response_times}")
print(f"Timestamps:\n{timestamps}")

Response times:
[120 135 145 150 155 160 165 170 500]
Timestamps:
[0 1 2 3 4 5 6 7 8]


In [ ]:
times = np.where(response_times > 200, 170, response_times)
print(f"Outlier:\n{times}")

Outlier:
[120 135 145 150 155 160 165 170 170]


Save both arrays to binary files (.tofile)

In [ ]:
times.tofile('times.bin')
timestamps.tofile('timestamps.bin')
memory = times.nbytes + timestamps.nbytes
print(f"Memory: {memory} bytes")

Memory: 72 bytes


In [ ]:
vec = times * 2 + 100
print(f"Vectorization:\n{vec}")
print(f"Memory: {vec.nbytes} bytes")

Vectorization:
[340 370 390 400 410 420 430 440 440]
Memory: 36 bytes


Compare with loop version time

In [ ]:
start = time.time()
result_loop_vec = []
for x in times:
    result_loop_vec.append(x * 2 + 100)
loop_time_vec = time.time() - start
print(f"Loop time for vec: {loop_time_vec:.4f}s")

start = time.time()
vec_calc = times * 2 + 100
vec_time_calc = time.time() - start
print(f"Vectorization time for vec: {vec_time_calc:.4f}s")

print(f"Speed up: {loop_time_vec / vec_time_calc:.2f}x faster")


Loop time for vec: 0.0001s
Vectorization time for vec: 0.0001s
Speed up: 1.60x faster
